# Population generation — multi-size pipeline with checkpoints

This notebook orchestrates the creation of synthetic populations for several target sizes,
with intermediate checkpoints in a `Temp/` folder located next to the notebook.

## Parameters

- `POPULATION_SIZES`: list of sizes to generate (multiples of 100)
- `FORCE_REGENERATE`: forces regeneration from eqasim (ignores the API cache)
- `FORCE_STEP`: forces a restart from a given step — `raw`, `fixed`, `pt_enriched`, `zone_enriched` or `scheduled`

The notebook goes **all the way**: it exports to `data/population/` (the only folder read
by GAMA and the agent server), sets the traits imputed from the EMC² microdata, and
returns a completeness verdict. Nothing to run by hand after it.

## Pipeline with checkpoints

| Step | Input | Output |
|---|---|---|
| 1 – eqasim generation | eqasim API | `Temp/1_raw/` |
| 2 – Activity validation | `Temp/1_raw/` | `Temp/2_fixed/` |
| 3 – PT enrichment | `Temp/2_fixed/` | `Temp/3_pt_enriched/` |
| 3bis – Zone enrichment (AAV2020 + density) | `Temp/3_pt_enriched/` | `Temp/4_zone_enriched/` |
| 4 – OSMnx itinerary computation | `Temp/4_zone_enriched/` | `Temp/5_scheduled/` |
| 6 – OSMnx warm-up | `Temp/5_scheduled/` | SQLite route cache |
| **7 – Final export** | `Temp/5_scheduled/` | **`data/population/`** |
| **8 – EMC² imputed traits** | `data/population/` | `data/population/` |
| **9 – Completeness audit** | `data/population/` | verdict complete / incomplete |

At each step, if the output file already exists in `Temp/`, the step is skipped.
To force a restart at a given step (and all the following ones), set `FORCE_STEP`.

## Prerequisites

| Service | Command | Port |
|---|---|---|
| eqasim | `docker compose up eqasim` | 8003 |

Python dependencies: `numpy`, `pandas`, `tqdm`, `osmnx`, `python-calamine`

### Required INSEE data (step 3bis)

Place in `data/insee/`:

| File | Source | Role |
|---|---|---|
| `fichier_diffusion_2026.xlsx` | [insee.fr/fr/statistiques/5040028](https://www.insee.fr/fr/statistiques/5040028) | Density grid 2025 — column `DENS_AAV` (required) |
| `AAV2020_au_01-01-2026.xlsx` | [insee.fr/fr/statistiques/5040879](https://www.insee.fr/fr/statistiques/5040879) | Pole/ring category per municipality (optional) |

The `DENS_AAV` column of the diffusion file already holds the density × AAV cross:
`1=Urbain dense` · `2=Urbain intermédiaire` · `3=Rural périurbain` · `4=Rural non périurbain`

**Step 3ter — Stratified selection (cohort sealing).** Optional (`SELECT_N`). Between
zone enrichment and routing: selects `SELECT_N` personas from the generated pool,
by proportional allocation to the survey's 12 ring × car-ownership cells
(`scripts/panel/seal_population.py select`), excludes homes outside the scope, and makes
the rest of the chain run on `toulouse_population_<SELECT_N>_<SELECT_TAG>.json`. The
final sealing (`seal`) is done outside the notebook, after step 9.


In [ ]:
import os, certifi
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
POPULATION_SIZES      = [1000]      # multiples of 100; can reach hundreds of thousands
GENERATE_PERSONALITY  = False           # True → generates the Big Five (slow)
FORCE_REGENERATE      = True           # True → calls eqasim with force=True + rewrites Temp/raw/
FORCE_STEP            = None            # None | 'raw' | 'fixed' | 'pt_enriched' | 'zone_enriched' | 'scheduled'
                                        # Forces a restart from this step (and all the following ones)

# ── Sampling frame (ticket 026) ───────────────────────────────────────────────
# The EMC² survey scope is a LIST OF MUNICIPALITIES, not a rectangle: it is the only
# way to say "no more, no less". PERIMETER=True serves it, and DEPARTMENTS restricts it.
#
# The study scope is that of the 453 municipalities across SIX departments (ticket 031, option A):
# DEPARTMENTS = None serves them all — provided that the BD TOPO and the BAN of departments 32,
# 81, 82, 09 and 11 are in eqasim-toulouse/data/ (otherwise the service refuses, code 3, with the
# list of what is missing). ['31'] = rehearsal on Haute-Garonne alone (346 municipalities): the
# 3rd ring caps there at 10.6 % of the population against 15.4 % in the survey
# (docs/arch/perimetre-population.md, limitation no. 6). Do not seal a v4 on this frame.
PERIMETER   = True
DEPARTMENTS = ['31']                      # None = the six departments of the scope

BBOX = None                               # ignored when PERIMETER=True; otherwise a WGS84 rectangle
# BBOX = [1.35, 43.55, 1.50, 43.65]      # example: central Toulouse

CLEAR_DOWNSTREAM_ON_REGENERATE = True    # True → deletes the downstream checkpoints when step 1 regenerates a raw

EQASIM_URL = 'http://localhost:8003'

# ── Stratified selection (cohort sealing) ──────────────────────────────────────
# The eqasim service draws 15 % more than requested and RENAMES the file to the requested
# size: "toulouse_population_1000.json" contains 1,021. For a round AND representative
# count, we generate a POOL (POPULATION_SIZES = [5000]) and select from it
# SELECT_N personas by proportional allocation to the survey's 12 ring × car-ownership
# cells (scripts/panel/seal_population.py select) — BEFORE the routing steps
# (4+5, 6), which grow with N and then only run on the retained ones. The rest of the
# chain (export, EMC² traits, audit) runs on toulouse_population_<SELECT_N>_<SELECT_TAG>.json.
# Pool: 5,000 gives a 99.2 % chance of filling the 12 cells; 2,700 gives 63 %.
# SELECT_N = None → no selection, the chain continues on the whole pool.
SELECT_N   = None                       # e.g. 1000
SELECT_TAG = 'PANEL'                    # suffix of the selected file
POPULATION_TAG = ''                     # set by step 3ter; do not edit by hand

# ── Routing: parallelism and warm-up ─────────────────────────────────────────
# Each worker loads its own copy of the OSMnx graphs (0.3 to 1.5 GB). At 12 workers the development
# machine swaps (measured on 2026-09-02: 23 GB of swap, workers at 50 % CPU) and
# step 6 takes hours; at 6 it fits in RAM. Step 6 does not touch the population:
# it pre-computes the itinerary cache to speed up runs. SKIP_WARMUP = True skips it
# — the runtime computes the missing itineraries on demand — and rerunning it later is
# done with SKIP_WARMUP = False, FORCE_STEP = None (the other steps are then skipped).
# Graph of the 453-municipality polygon (ticket 031 § 1.4): ≈ 1 GB per loaded worker (measured, cf.
# docs/traces/<date>_graphe_osmnx_perimetre_453); 6 workers fit in 32 GB with the rest
# of the machine, 12 swap. The graph is built once: `make osmnx-perimeter-graph`.
MAX_WORKERS = 6
SKIP_WARMUP = True                      # default: skipped — rerun it when a run needs it (decision 2026-09-02)
SELECTION_RULE = 'panel_seal_v4'        # selection rule expected at step 3ter (scripts/panel/seal_population.py)


## Environment initialisation

Creation of the temporary folder tree (`Temp/`) and definition of the paths to the source and output data. Each step writes into a numbered subfolder (`1_raw/`, `2_fixed/`, …) so that the run can resume from any point without recomputing everything.

The function `should_force(step)` decides whether a step must be replayed, taking `FORCE_REGENERATE` and `FORCE_STEP` into account.

In [ ]:
# ── Paths & Temp folders ──────────────────────────────────────────────────────
import json
import os
import sys
import time
import urllib.request
import urllib.error
from pathlib import Path

REPO_ROOT    = Path('../../../').resolve()
POP_DIR      = REPO_ROOT / 'data' / 'population'
NOTEBOOK_DIR = Path('.').resolve()
TEMP_DIR     = NOTEBOOK_DIR / 'Temp'

TEMP_RAW       = TEMP_DIR / '1_raw'
TEMP_FIXED     = TEMP_DIR / '2_fixed'
TEMP_PT        = TEMP_DIR / '3_pt_enriched'
TEMP_ZONE      = TEMP_DIR / '4_zone_enriched'
TEMP_SCHEDULED = TEMP_DIR / '5_scheduled'

for d in [POP_DIR, TEMP_RAW, TEMP_FIXED, TEMP_PT, TEMP_ZONE, TEMP_SCHEDULED]:
    d.mkdir(parents=True, exist_ok=True)

# ── Utilities ─────────────────────────────────────────────────────────────────
STEPS = ['raw', 'fixed', 'pt_enriched', 'zone_enriched', 'scheduled']

def should_force(step_name: str) -> bool:
    if FORCE_REGENERATE and step_name == 'raw':
        return True
    if FORCE_STEP is None or FORCE_STEP not in STEPS:
        return False
    return STEPS.index(step_name) >= STEPS.index(FORCE_STEP)

def pop_filename(n: int) -> str:
    # The suffix (POPULATION_TAG) is set by step 3ter after selection: the selected
    # population has its own name and never overwrites toulouse_population_<n>.json.
    tag = f'_{POPULATION_TAG}' if POPULATION_TAG else ''
    return f'toulouse_population_{n}{tag}.json'

def save_json(data, path: Path) -> None:
    tmp = path.with_suffix('.json.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.rename(tmp, path)

def load_json(path: Path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

def check_temporal_order(data: list) -> tuple[int, int]:
    """Check start_time ordering (at most 1 decreasing gap allowed for midnight wrap).
    Returns (n_persons_with_error, n_total_errors).
    """
    n_persons, n_total = 0, 0
    for person in data:
        acts = person.get('identity', {}).get('activities', [])
        n = len(acts)
        if n == 0:
            continue
        if n == 1:
            # IMMOBILE day (ticket 029): a single activity, home 0 → 86,400 s. It
            # is valid by construction; the "zero duration" test below rejected it
            # (86,400 mod 86,400 = 0) and the export refused any population with immobile persons.
            continue
        errors = []
        for i, act in enumerate(acts):
            s, e = act.get('start_time'), act.get('end_time')
            if s is not None and e is not None and (e - s) % 86400 == 0:
                errors.append(f"act[{i}] durée nulle")
        nb_ecarts = sum(
            1 for i in range(n)
            if acts[(i - 1) % n].get('start_time', 0) > acts[i].get('start_time', 0)
        )
        if nb_ecarts > 1:
            errors.append(f"{nb_ecarts} écarts décroissants sur start_time")
        if errors:
            n_persons += 1
            n_total += len(errors)
    return n_persons, n_total

print(f'REPO_ROOT : {REPO_ROOT}')
print(f'POP_DIR   : {POP_DIR}')
print(f'TEMP_DIR  : {TEMP_DIR}')
print(f'Sizes     : {POPULATION_SIZES}')

## Loading the scientific dependencies

Import of the numerical computing libraries (`numpy`, `pandas`) and configuration of the constants shared between steps:

| Constant | Value | Role |
|---|---|---|
| `GTFS_FEED_PRIMAIRE` | `data/gtfs/tisseo_gtfs/` | Enumeration root of the GTFS feeds **in service** (Tisséo, TER, liO regional coach) for PT enrichment — the same as the runtime proximity gate |
| `OSMNX_CACHE` | `data/cache/osmnx/` | Road graph cache (avoids re-downloading) |
| `MAX_WORKERS` | 12 | Parallelism of route computation |
| `MAX_PT_DIST_M` | 1,500 m | Radius for attaching to a PT stop |

In [ ]:
import re
# ── Scientific imports & constants ───────────────────────────────────────────
import hashlib
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# PRIMARY feed, used as the enumeration root: step 3 reads the stops of ALL the
# feeds in service next to it (Tisséo, TER, liO regional coach), as the runtime
# proximity gate does. Before 2026-09-04 it read only Tisséo, and 397 of the 2,580
# points of population v4 — including 245 of the 374 in the 3rd ring — were declared without
# service although a regional stop or a station is less than 1,500 m away.
GTFS_FEED_PRIMAIRE = REPO_ROOT / 'data' / 'gtfs' / 'tisseo_gtfs'
OSMNX_CACHE       = REPO_ROOT / 'data' / 'cache' / 'osmnx'      # graphs (pickle) — local notebook
OSMNX_ROUTE_CACHE = REPO_ROOT / 'data' / 'cache' / 'osmnx'             # SQLite routes — SAME file as the runtime
SCRIPTS_POP    = REPO_ROOT / 'scripts' / 'data' / 'population'
LLMAGENTS_PATH = str(REPO_ROOT / 'services' / 'llm-agents')

# Routing graph for steps 4+5: the 453-municipality polygon (ticket 031 § 1.4), built by
# scripts/data/population/build_osmnx_perimeter_graph.py — a key distinct from the 30 km disc of
# production (`Toulouse, France_30000` → ecb40f20a303). Without this graph, 98 of the 154
# 3rd-ring agents of v3 were routed onto the same node and received a fallback speed.
from build_osmnx_perimeter_graph import PERIMETER_CACHE_KEY, PERIMETER_GRAPH_LABEL
CACHE_KEY     = PERIMETER_CACHE_KEY
if not (OSMNX_CACHE / f'graphs_{CACHE_KEY}.pkl').exists():
    raise FileNotFoundError(
        f'Scope OSMnx graph missing: {OSMNX_CACHE}/graphs_{CACHE_KEY}.pkl '
        f'({PERIMETER_GRAPH_LABEL}). Build it: make osmnx-perimeter-graph')
# MAX_WORKERS: parameter of the first cell (papermill can override it)
MAX_PT_DIST_M = 1500.0

if LLMAGENTS_PATH not in sys.path:
    sys.path.insert(0, LLMAGENTS_PATH)
if str(SCRIPTS_POP) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_POP))

from population_utils import TRIP_MODES

print(f'GTFS_FEED_PRIMAIRE:  {GTFS_FEED_PRIMAIRE}')
print(f'OSMNX_CACHE: {OSMNX_CACHE}')
print(f'OSMnx cache key: {CACHE_KEY}  ({PERIMETER_GRAPH_LABEL})')
print(f'OSMnx route SQLite: {OSMNX_ROUTE_CACHE}')

# ── Safeguard: the populator and the runtime must write the SAME file ─────────
# From 2026-06-02 to 2026-09-04, they wrote into two distinct files: the
# step 6 warm-up (83,478 routes) never served a single run. The alignment
# rests on a docker mount; we check that it is still there rather than trusting it.
_runtime_dir = re.search(
    r'osmnx_persistent_cache_dir:\s*str\s*=\s*"([^"]+)"',
    (REPO_ROOT / 'services' / 'llm-agents' / 'settings.py').read_text(),
).group(1)
_expected_mount = f"./{OSMNX_ROUTE_CACHE.relative_to(REPO_ROOT).as_posix()}:{_runtime_dir}"
if _expected_mount not in (REPO_ROOT / 'infra' / 'docker-compose.yml').read_text():
    raise RuntimeError(
        f"Route cache misaligned: the runtime reads {_runtime_dir}, but docker-compose.yml "
        f"does not mount `{_expected_mount}`. Warming up here would fill a file that "
        f"nobody reads — fix the mount (controller service) before continuing."
    )
print(f'✓ route cache shared with the runtime: {_expected_mount}')


## eqasim service check

Before any processing, we make sure that the eqasim population generation service answers on `EQASIM_URL` (port 8003). If it fails after 3 attempts, the pipeline stops immediately with an explicit error message.

> **Prerequisite**: `docker compose up eqasim` must be running before executing this cell.

In [ ]:
# ── eqasim service health check ──────────────────────────────────────────────
def check_health(url: str, retries: int = 3, delay: float = 2.0) -> bool:
    for i in range(retries):
        try:
            with urllib.request.urlopen(f'{url}/health', timeout=5) as r:
                return r.status == 200
        except Exception as e:
            print(f'  attempt {i+1}/{retries}: {e}')
            if i < retries - 1:
                time.sleep(delay)
    return False

# The service is only called at step 1, and only if a raw pool is missing or if
# regeneration is forced: a pool already in Temp/1_raw/ (generated by the service, then Docker
# stopped to give its memory back to routing) does not require it.
_needs_eqasim = FORCE_REGENERATE or any(not (TEMP_RAW / pop_filename(n)).exists() for n in POPULATION_SIZES)
if not _needs_eqasim:
    print(f'[SKIP] eqasim service not called: the raw pools {POPULATION_SIZES} are in Temp/1_raw/')
elif check_health(EQASIM_URL):
    print(f'eqasim service OK → {EQASIM_URL}')
else:
    raise RuntimeError(
        f'eqasim service not responding on {EQASIM_URL}. '
        'Start the service with: docker compose up eqasim'
    )


---
## Step 1 — eqasim generation → `Temp/1_raw/`

Call to the eqasim API for each population size defined in `POPULATION_SIZES`. eqasim synthesises Toulouse agents with their daily activities (home, work, leisure…) from INSEE data and travel surveys (EMC²).

- **Cache**: if the file already exists in `Temp/1_raw/` and `FORCE_REGENERATE=False`, the API call is skipped.
- **Output**: one JSON file per size containing the list of persons with their raw activities and their socio-demographic attributes.
- **Validation**: after generation, the temporal order of activities is checked — at most one decreasing gap is tolerated (midnight crossing).

**Sampling frame (ticket 026).** With `PERIMETER = True`, eqasim no longer draws in a
rectangle or in a whole department but in the **list of municipalities of the EMC²
survey scope** — 346 municipalities with `DEPARTMENTS = ['31']`, 453 with `DEPARTMENTS = None`.
The `sampling_rate` is recomputed on the RP 2022 population of these municipalities.

⚠ With `DEPARTMENTS = ['31']`, the chain is a **rehearsal** on Haute-Garonne: the
107 municipalities of the five other departments of the scope require their BD TOPO and their BAN
(ticket 031 § 1.0, agreement of the repository author). The limitation is quantified and published —
`docs/arch/perimetre-population.md`, limitation no. 6: the 3rd ring caps at 10.6 % of the
population instead of 15.4 %. The eqasim service logs the frame retained per department and
refuses to generate if a requested department does not have its data.


In [ ]:
# ── Step 1 — eqasim generation → Temp/1_raw/ ──────────────────────────────────
print('=' * 60)
print('STEP 1 — eqasim generation → Temp/1_raw/')
print('=' * 60)

for pop_size in POPULATION_SIZES:
    fname    = pop_filename(pop_size)
    raw_path = TEMP_RAW / fname

    if not should_force('raw') and raw_path.exists():
        size_mb = raw_path.stat().st_size / 1_048_576
        print(f'[SKIP] {fname}  ({size_mb:.1f} MB) — already in Temp/1_raw/')
        continue

    payload = {
        'population_size':      pop_size,
        'generate_personality': GENERATE_PERSONALITY,
        'force':                FORCE_REGENERATE,
    }
    if PERIMETER:
        # Takes precedence over BBOX server-side: a rectangle cannot express the scope.
        payload['perimeter'] = True
        if DEPARTMENTS is not None:
            payload['departments'] = DEPARTMENTS
    elif BBOX is not None:
        payload['bbox'] = BBOX

    body = json.dumps(payload).encode()
    req  = urllib.request.Request(
        f'{EQASIM_URL}/generate',
        data=body,
        headers={'Content-Type': 'application/json'},
        method='POST',
    )

    print(f'[GEN]  {fname}  (population_size={pop_size})…')
    t0 = time.monotonic()

    try:
        with urllib.request.urlopen(req, timeout=7200) as resp:
            result = json.loads(resp.read())
    except urllib.error.HTTPError as e:
        result = json.loads(e.read())
        raise RuntimeError(f'eqasim HTTP {e.code} : {result}')

    elapsed = time.monotonic() - t0
    if result.get('status') != 'ok':
        raise RuntimeError(f'eqasim failure for {pop_size} agents: {result}')

    eqasim_out = POP_DIR / fname
    if not eqasim_out.exists():
        raise FileNotFoundError(f'eqasim file not found: {eqasim_out}')

    data = load_json(eqasim_out)
    save_json(data, raw_path)
    size_mb = raw_path.stat().st_size / 1_048_576
    print(f'       {len(data)} persons in {elapsed:.1f}s  ({size_mb:.1f} MB) → Temp/1_raw/{fname}')
    if CLEAR_DOWNSTREAM_ON_REGENERATE:
        # TEMP_ZONE included: without it, step 3bis "skipped" on the pre-imputed checkpoint of
        # the old pool and the selection restarted from a population that step 1 had just
        # replaced (observed on 2026-09-03 on pool v3 → v4).
        for _dl_dir in [TEMP_FIXED, TEMP_PT, TEMP_ZONE, TEMP_SCHEDULED]:
            _p = _dl_dir / fname
            if _p.exists():
                _p.unlink()
                print(f'       [CASCADE] Deleted {_dl_dir.name}/{fname}')
        _sqlite = OSMNX_CACHE / Path(fname).stem / 'osmnx_cache.db'
        if _sqlite.exists():
            _sqlite.unlink()
            print(f'       [CASCADE] Deleted OSMnx SQLite cache: {_sqlite.relative_to(REPO_ROOT)}')
    n_p, n_e = check_temporal_order(data)
    print(f'       {"[OK]  " if n_e == 0 else "[WARN]"} Temporal order: {len(data) - n_p}/{len(data)} valid persons')

print()
print('Step 1 finished.')

---
## Step 2 — Validation & correction of activities → `Temp/2_fixed/`

Raw eqasim data can contain invalid activity sequences (overlaps, zero durations, inconsistent chains). This step applies two cleaning passes:

1. **Correction of sequence violations** via `fix_activities`: readjustment of the time bounds to remove overlaps and negative durations.
2. **Merge of redundant activities**: two consecutive (or circular) activities with the same `purpose` and the same location are merged into one, to avoid zero-duration micro-trips.

Each agent is re-validated after correction; cases that are still invalid are reported but do not interrupt the pipeline.

**Activities outside the 453-municipality polygon** (ticket 031, assumed hypothesis): an activity
located outside the survey scope is **removed** from the chain (never the home), counted at
the root of the record (`perimetre.activites_hors_perimetre_supprimees`) and alarmed if the
count exceeds 0; the seal's MANIFEST reports the total. Measured: 0 on the pools of 2026-09-03.

In [ ]:
# ── Step 2 — Validation & correction of activities → Temp/2_fixed/ ───────────
from population_utils import check_activities, fix_activities, drop_out_of_perimeter_activities

print('=' * 60)
print('STEP 2 — Validation & correction of activities → Temp/2_fixed/')
print('=' * 60)

# Activities outside the 453-municipality polygon (ticket 031, decision of 2026-09-03): removed from
# the chain BEFORE the schedule readjustment, counted at the root of the record, alarmed.
# The geometry is the survey's (couronne_perimetre.geojson), the same as for the
# residence_zone trait and for audit A4.
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from mobility_core.residence_zone import CommunalZones
_ZONES = CommunalZones.load()
_classify_perimetre = lambda lat, lon: _ZONES.classify(lat, lon) or ''

def _same_loc(a, b):
    la, lb = a.get('location'), b.get('location')
    if la is None or lb is None:
        return la is lb
    return la.get('lon') == lb.get('lon') and la.get('lat') == lb.get('lat')

for pop_size in POPULATION_SIZES:
    fname      = pop_filename(pop_size)
    raw_path   = TEMP_RAW   / fname
    fixed_path = TEMP_FIXED / fname

    if not should_force('fixed') and fixed_path.exists():
        print(f'[SKIP] {fname} — already in Temp/2_fixed/')
        continue

    if not raw_path.exists():
        raise FileNotFoundError(f'Raw file missing — rerun step 1: {raw_path}')

    data = load_json(raw_path)

    # Out-of-scope activities: removal + count (0 expected; > 0 raises an alarm)
    _hors, _touches = 0, 0
    for person in data:
        _k = drop_out_of_perimeter_activities(person, _classify_perimetre)
        _hors += _k
        _touches += 1 if _k else 0
    if _hors:
        print(f'[ALARME] {fname}: {_hors} activity(ies) outside the 453-municipality polygon removed '
              f'for {_touches} person(s) — assumed hypothesis (ticket 031), reported in the MANIFEST')
    else:
        print(f'[OK]   {fname} — 0 activities outside the 453-municipality polygon ({len(data)} persons checked)')

    # Correction of sequence violations
    persons_fixed = 0
    fixed_data = []
    for person in data:
        fixed_person, _ = fix_activities(person)
        remaining = check_activities(fixed_person)
        if remaining:
            print(f'  !! person {person.get("person_id", "?")}: still invalid after correction: {remaining}')
        fixed_data.append(fixed_person)
        persons_fixed += 1

    # Merge: consecutive AND circular activities sharing the same purpose AND the same location
    n_merged = 0
    for person in fixed_data:
        acts = person.get('identity', {}).get('activities', [])

        # Consecutive merge
        i = 0
        while i < len(acts) - 1:
            if acts[i].get('purpose') == acts[i + 1].get('purpose') and _same_loc(acts[i], acts[i + 1]):
                acts[i]['end_time'] = acts[i + 1]['end_time']
                acts[i]['scheduled_start_time'] = None
                acts.pop(i + 1)
                n_merged += 1
            else:
                i += 1

        # Circular merge: first and last
        if len(acts) >= 2 and acts[0].get('purpose') == acts[-1].get('purpose') and _same_loc(acts[0], acts[-1]):
            last = acts.pop()
            acts[0]['start_time'] = last['start_time']
            acts[0]['scheduled_start_time'] = None
            n_merged += 1

    save_json(fixed_data, fixed_path)
    print(f'[OK]   {fname} — {persons_fixed} corrected, {n_merged} merge(s) → Temp/2_fixed/')
    n_p, n_e = check_temporal_order(fixed_data)
    print(f'       {"[OK]  " if n_e == 0 else "[WARN]"} Temporal order: {len(fixed_data) - n_p}/{len(fixed_data)} valid persons')

print()
print('Step 2 finished.')

---
## Step 3 — Public transport enrichment → `Temp/3_pt_enriched/`

For each activity location, we compute whether it lies within `MAX_PT_DIST_M` (1,500 m) of a Tisséo stop. This `public_transport` flag is then used by the LLM agent to decide on the relevant travel mode (walking, car, PT…).

- **Source**: `stops.txt` file of the Tisséo GTFS — 5,661 stops loaded in memory.
- **Method**: nearest-stop search by approximate Euclidean distance on the WGS84 coordinates.
- The number of enriched locations is shown per file.

In [ ]:
# ── Step 3 — Enrichment of the public_transport flags → Temp/3_pt_enriched/ ─
from population_utils import enrich_public_transport

print('=' * 60)
print('STEP 3 — public_transport enrichment → Temp/3_pt_enriched/')
print('=' * 60)

# The stops of the THREE networks in service, enumerated by the SAME function as the runtime
# proximity gate (`trip_helper.otp.feeds_en_service`): the flag set here and the
# filter applied at runtime must talk about the same stops, otherwise a persona declared
# as served is refused OTP, or the reverse.
from trip_helper.otp import feeds_en_service

_feeds = feeds_en_service(str(GTFS_FEED_PRIMAIRE))
if not _feeds:
    raise FileNotFoundError(
        f'No GTFS feed found next to {GTFS_FEED_PRIMAIRE}: the '
        f'public_transport flag would be wrong for the whole population.')
_par_feed, _frames = {}, []
for _feed in _feeds:
    _f = pd.read_csv(_feed / 'stops.txt', usecols=['stop_lat', 'stop_lon']) \
        if _feed.is_dir() else None
    if _f is None:
        print(f'[WARN] {_feed.name} skipped (zip archive not read at this step)')
        continue
    _par_feed[_feed.name] = len(_f)
    _frames.append(_f)
stops_df  = pd.concat(_frames, ignore_index=True).dropna(subset=['stop_lat', 'stop_lon'])
stop_lats = stops_df['stop_lat'].values
stop_lons = stops_df['stop_lon'].values
print(f'Chargement GTFS : {len(stops_df)} arrêts sur {len(_par_feed)} réseau(x) — {_par_feed}')
if len(_par_feed) < 2:
    print('[ALARME] un seul réseau GTFS lu : le drapeau de desserte sous-estimera la '
          '2ᵉ et la 3ᵉ couronne, comme avant le 2026-09-04')
print()

for pop_size in POPULATION_SIZES:
    fname      = pop_filename(pop_size)
    fixed_path = TEMP_FIXED / fname
    pt_path    = TEMP_PT    / fname

    if not should_force('pt_enriched') and pt_path.exists():
        print(f'[SKIP] {fname} — already in Temp/3_pt_enriched/')
        continue

    if not fixed_path.exists():
        raise FileNotFoundError(f'Fixed file missing — rerun step 2: {fixed_path}')

    data = load_json(fixed_path)
    n = enrich_public_transport(data, stop_lats, stop_lons, MAX_PT_DIST_M)
    save_json(data, pt_path)
    print(f'[OK]   {fname} — {n} location(s) enriched → Temp/3_pt_enriched/')
    n_p, n_e = check_temporal_order(data)
    print(f'       {"[OK]  " if n_e == 0 else "[WARN]"} Temporal order: {len(data) - n_p}/{len(data)} valid persons')

print()
print('Step 3 finished.')

---
## Step 3bis — Urban/peri-urban/rural zone enrichment → `Temp/4_zone_enriched/`

For each activity location, the zone type is determined from two INSEE tables:

| Source | Role |
|---|---|
| **AAV 2020** (Zonage en Aires d'Attraction des Villes) | Pole / ring / outside-influence classification |
| **Municipal density grid** | Density level 1–4 (dense → very sparse) |

**Pipeline**:
1. Collection of all the unique coordinates of the population
2. Batch reverse geocoding via the BAN API (`api-adresse.data.gouv.fr`) → INSEE municipality code
3. Join with AAV2020 + density grid
4. Addition of the `zone_type` field in each activity `location`

**Possible values of `zone_type`**:

| Value | Meaning |
|---|---|
| `urbain_dense` | AAV pole + dense or intermediate municipality |
| `urbain` | AAV pole + sparse municipality |
| `periurbain_dense` | AAV ring + dense or intermediate municipality |
| `periurbain` | Ring/multipolarised + sparse municipality |
| `bourg` | Outside AAV influence + dense municipality |
| `rural` | Outside AAV influence + sparse municipality |
| `inconnu` | Municipality code not found in the INSEE tables |

> **Prerequisite**: place the files `AAV2020_au_01-01-2023.csv` and `grille_communale_densite_2023.csv` in `data/insee/` (see the section "Required INSEE data" in the introduction).

In [ ]:
# ── Step 3bis — Zone enrichment (AAV2020 + density) → Temp/4_zone_enriched/ ─
import csv
import io
import ssl
import time as _time

INSEE_DIR          = REPO_ROOT / 'data' / 'insee'
DENSITE_FILE       = INSEE_DIR / 'fichier_diffusion_2026.xlsx'
AAV_FILE           = INSEE_DIR / 'AAV2020_au_01-01-2026.xlsx'
GEOCODE_CACHE_FILE = TEMP_DIR  / 'geocode_cache.json'

_SSL_CTX = ssl.create_default_context()
_SSL_CTX.check_hostname = False
_SSL_CTX.verify_mode    = ssl.CERT_NONE

print('=' * 60)
print('STEP 3bis — Zone enrichment → Temp/4_zone_enriched/')
print('=' * 60)

if should_force('zone_enriched') and GEOCODE_CACHE_FILE.exists():
    GEOCODE_CACHE_FILE.unlink()
    print('[RESET] Geocoding cache deleted (FORCE_STEP)')

if not DENSITE_FILE.exists():
    raise FileNotFoundError(
        f'Density file missing: {DENSITE_FILE}\n'
        f'Download it from https://www.insee.fr/fr/statistiques/5040028\n'
        f'and place it in {INSEE_DIR}/'
    )

# ── Loading the density grid (DENS7) ────────────────────────────────────────
densite = pd.read_excel(
    DENSITE_FILE, sheet_name='Maille communale', header=4,
    dtype={'CODGEO': str}, engine='calamine',
)
densite.columns = [c.strip() for c in densite.columns]
print(f'Density grid: {len(densite)} municipalities')

# Natural labels for DENS7 — used in the final sentence
# Labels of the INSEE density grid (DENS7), in ENGLISH since ticket 074: this
# string is composed by US and read by the model in its prompt (`destination_zone`).
# Not to be confused with survey NOMENCLATURES (EMC² rings, CEREMA occupation), which stay
# in the language of the source they cite and are translated at display time.
DENS7_LABEL = {
    1: 'a major urban centre',
    2: 'an intermediate urban centre',
    3: 'an urban belt',
    4: 'a small town',
    5: 'a rural small town',
    6: 'scattered rural housing',
    7: 'very scattered rural housing',
}
densite['_detail'] = densite['DENS7'].map(DENS7_LABEL).fillna('an unknown area')
_densite_lookup    = densite.set_index('CODGEO')['_detail'].to_dict()
# The MUNICIPALITY name, which was missing from the label read by the model (cf. `_build_zone_label`).
_densite_lookup_libgeo = densite.set_index('CODGEO')['LIBGEO'].to_dict()

print('DENS7 distribution:')
print(densite['_detail'].value_counts().to_string())
print()

# ── Loading AAV2020 — central city ────────────────────────────────────────────
# MUNICIPALITY NAME and AREA NAME are two different columns, and confusing them was the
# defect fixed on 2026-09-04: `LIBAAV2020` is the label of the CATCHMENT AREA, which is
# "Toulouse" for the four hundred or so municipalities of the area. Written in place of the
# municipality name, it made the model read « quartier de bourg rural sur la commune de Toulouse » for
# an activity 70 km from the Capitole — 2,073 of the 3,383 labels of population v4, i.e. 61 %.
# `LIBGEO` holds the municipality name, in both files.
_commune_lookup: dict[str, str] = dict(_densite_lookup_libgeo)
_attraction_lookup: dict[str, str] = {}
_hors_aire: set = set()
if AAV_FILE.exists():
    try:
        aav = pd.read_excel(
            AAV_FILE, sheet_name='Composition_communale',
            header=5, dtype={'CODGEO': str}, engine='calamine',
        )
        aav.columns = [c.strip() for c in aav.columns]
        # `CATEAAV2020` is 30 for « commune hors attraction des villes », and it is read as
        # an INTEGER: the comparison `!= '30'` before 2026-09-04 was therefore always true,
        # and these municipalities received « sur la commune de Commune hors attraction des villes ».
        _cat = pd.to_numeric(aav['CATEAAV2020'], errors='coerce')
        _hors_aire = set(aav.loc[_cat == 30, 'CODGEO'])
        _attraction_lookup = aav.loc[_cat != 30].set_index('CODGEO')['LIBAAV2020'].to_dict()
        for _code, _nom in aav.set_index('CODGEO')['LIBGEO'].to_dict().items():
            _commune_lookup.setdefault(_code, _nom)
        print(f'AAV2020: {len(aav)} municipalities, of which {len(_hors_aire)} outside a catchment area')
    except Exception as e:
        print(f'[WARN] AAV2020 not loaded ({e})')
else:
    print('[INFO] AAV2020 missing — catchment area ignored')
if not _commune_lookup:
    raise RuntimeError(
        'No municipality name loaded: the zone label read by the model would say "an unknown '
        'municipality" for the whole population. Check the LIBGEO column of '
        f'{DENSITE_FILE.name} and of {AAV_FILE.name}.')
print()

_libelles_sans_commune: set = set()


def _build_zone_label(codgeo: str) -> str:
    """The zone label READ BY THE MODEL in its prompt. The municipality in it is the real one.

    Examples:
      31555 → 'a neighbourhood of a major urban centre in the municipality of Toulouse'
      31561 → 'a neighbourhood of an intermediate urban centre in the municipality of L'Union (Toulouse urban area)'
      09001 → 'a neighbourhood of scattered rural housing in the municipality of Aigues-Juntes, outside any urban catchment area'

    Two defects fixed on 2026-09-04, both in the same expression: the named municipality
    was actually the CATCHMENT AREA ("Toulouse" for four hundred municipalities), and the test
    meant to exclude municipalities outside any area compared an integer with a string, so excluded
    nothing — hence labels « sur la commune de Commune hors attraction des villes ».
    """
    label   = _densite_lookup.get(codgeo, 'an unknown area')
    commune = _commune_lookup.get(codgeo)
    aire    = _attraction_lookup.get(codgeo, '')
    if commune is None:
        # We do not make up a municipality name: "unknown" is information, an invented
        # name is not. The count is reported at the end of the step.
        _libelles_sans_commune.add(codgeo)
        lieu = 'an unknown municipality'
    else:
        # The municipality NAME stays French — it is a proper name, invariant across languages.
        # Only the frame of the sentence switches (ticket 074).
        lieu = f'the municipality of {commune}'
    if aire:
        return f'a neighbourhood of {label} in {lieu} ({aire} urban area)'
    return f'a neighbourhood of {label} in {lieu}, outside any urban catchment area'

# ── Batch reverse geocoding (BAN API) — multipart/form-data ───────────────────
def _make_multipart(csv_bytes: bytes) -> tuple[bytes, str]:
    boundary = f'----BoundaryZone{int(_time.time())}'
    body = (
        f'--{boundary}\r\n'
        f'Content-Disposition: form-data; name="data"; filename="coords.csv"\r\n'
        f'Content-Type: text/csv\r\n\r\n'
    ).encode() + csv_bytes + f'\r\n--{boundary}--\r\n'.encode()
    return body, f'multipart/form-data; boundary={boundary}'

_geocode_cache: dict[tuple, str] = {}
if GEOCODE_CACHE_FILE.exists():
    _raw = load_json(GEOCODE_CACHE_FILE)
    _geocode_cache = {(float(k.split(',')[0]), float(k.split(',')[1])): v
                      for k, v in _raw.items()}
    print(f'Geocoding cache: {len(_geocode_cache)} coords')

def _save_geocode_cache() -> None:
    save_json({f'{k[0]},{k[1]}': v for k, v in _geocode_cache.items()}, GEOCODE_CACHE_FILE)

def _batch_reverse_geocode(coords: list[tuple[float, float]]) -> dict[tuple, str]:
    BATCH   = 5000
    result  = dict(_geocode_cache)
    missing = [c for c in coords if c not in result]
    if not missing:
        return result
    print(f'  {len(missing)} coords to geocode ({len(coords) - len(missing)} cached)…')
    for start in range(0, len(missing), BATCH):
        chunk     = missing[start:start + BATCH]
        csv_bytes = ('longitude,latitude\n' + '\n'.join(f'{lon},{lat}' for lon, lat in chunk)).encode()
        body, ctype = _make_multipart(csv_bytes)
        req = urllib.request.Request(
            'https://api-adresse.data.gouv.fr/reverse/csv/',
            data=body, headers={'Content-Type': ctype}, method='POST',
        )
        try:
            with urllib.request.urlopen(req, timeout=120, context=_SSL_CTX) as resp:
                reader = csv.DictReader(io.TextIOWrapper(resp, encoding='utf-8'))
                for row in reader:
                    try:
                        key = (float(row['longitude']), float(row['latitude']))
                        result[key] = row.get('result_citycode', '')
                        _geocode_cache[key] = result[key]
                    except (KeyError, ValueError):
                        pass
        except Exception as e:
            print(f'  [WARN] BAN API error (chunk {start}): {e}')
        _save_geocode_cache()
        print(f'  Geocoded {min(start + BATCH, len(missing))}/{len(missing)} coords…')
    n_ok = sum(1 for v in result.values() if v)
    print(f'  {n_ok}/{len(coords)} municipality codes found')
    return result

_STALE_FIELDS = {'zone_city', 'zone_density', 'zone_type', 'zone_type_detail',
                 'commune_pop', 'attraction_city', 'aav_category'}

# ── Processing per population size ────────────────────────────────────────────
for pop_size in POPULATION_SIZES:
    fname     = pop_filename(pop_size)
    pt_path   = TEMP_PT   / fname
    zone_path = TEMP_ZONE / fname

    if not should_force('zone_enriched') and zone_path.exists():
        print(f'[SKIP] {fname} — already in Temp/4_zone_enriched/')
        continue

    if not pt_path.exists():
        raise FileNotFoundError(f'pt_enriched file missing — rerun step 3: {pt_path}')

    data = load_json(pt_path)

    unique_locs = list({
        (act['location']['lon'], act['location']['lat'])
        for person in data
        for act in person.get('identity', {}).get('activities', [])
        if act.get('location') and act['location'].get('lon') is not None
    })
    print(f'{fname}: {len(unique_locs)} unique locations to geocode…')

    loc_to_codgeo = _batch_reverse_geocode(unique_locs)
    loc_to_zone   = {loc: _build_zone_label(codgeo) for loc, codgeo in loc_to_codgeo.items()}

    n_enriched = 0
    for person in data:
        for act in person.get('identity', {}).get('activities', []):
            loc = act.get('location')
            if not loc or loc.get('lon') is None:
                continue
            for f in _STALE_FIELDS:
                loc.pop(f, None)
            loc['zone'] = loc_to_zone.get((loc['lon'], loc['lat']), 'an unknown area')
            n_enriched += 1

    save_json(data, zone_path)

    zone_counts: dict[str, int] = {}
    for person in data:
        for act in person.get('identity', {}).get('activities', []):
            z = act.get('location', {}).get('zone', 'an unknown area')
            zone_counts[z] = zone_counts.get(z, 0) + 1
    print(f'[OK]   {fname} — {n_enriched} activities enriched → Temp/4_zone_enriched/')
    top5 = sorted(zone_counts.items(), key=lambda x: -x[1])[:5]
    print(f'       Top 5 zones: {top5}')

print()
print('Step 3bis finished.')

---

## Step 3ter — Stratified selection (cohort sealing)

Only runs if `SELECT_N` is set. Takes the pool from `Temp/4_zone_enriched/`, sets the
residence ring on each persona if it is still missing (same path as
`make residence-zone`), excludes homes outside the 453 municipalities and children under 5, then
retains `SELECT_N` personas by proportional allocation to the joint ring ×
car-ownership target (person base, `scripts/panel/cible_jointe_couronne_motorisation.yaml`). Deterministic
within-cell order (`sha256`), log in `*_selection.json`.

A **deficit** (a cell the pool does not fill) is made up from the same ring, then
from the whole pool, logged, and returns code 1: the chain continues, but the final
check will see it. After this step, `POPULATION_SIZES` and `POPULATION_TAG` designate the selected
file: steps 4 to 9 only run on it.


In [ ]:
# ── Step 3ter — stratified selection → Temp/4_zone_enriched/ ───────────
import subprocess

print('=' * 60)
print('STEP 3ter — Stratified selection (cohort sealing)')
print('=' * 60)

if not SELECT_N:
    print('SELECT_N = None — no selection, the chain continues on the whole pool.')
else:
    # Post-processing interpreter: the selection imports `mobility_core` (shapely,
    # pyproj, geopandas) and `scripts.*` — same choice as in step 8.
    _venv = REPO_ROOT / 'services' / 'llm-agents' / '.venv' / 'bin' / 'python'
    _select_python = str(_venv) if _venv.exists() else sys.executable
    _env = {**os.environ, 'PYTHONPATH': str(REPO_ROOT)}

    for pop_size in POPULATION_SIZES:
        pool = TEMP_ZONE / pop_filename(pop_size)
        if not pool.exists():
            raise FileNotFoundError(f'Pool missing — rerun step 3bis: {pool}')
        out = TEMP_ZONE / f'toulouse_population_{SELECT_N}_{SELECT_TAG}.json'
        # An existing selection is only reused if it comes from THIS pool: the log
        # holds the pool's sha256, we compare it. Otherwise (other pool, other rule), we redo it —
        # a stale selection served silently once caused a v2 population to be exported under a
        # v3 name (2026-09-03).
        _journal = out.with_name(out.stem + '_selection.json')
        _same_pool = False
        if out.exists() and _journal.exists():
            _pool_sha = hashlib.sha256(pool.read_bytes()).hexdigest()
            try:
                _prev = json.loads(_journal.read_text(encoding='utf-8'))
                _same_pool = _prev.get('vivier', {}).get('sha256') == _pool_sha and _prev.get('version') == SELECTION_RULE
            except (OSError, ValueError):
                _same_pool = False
        if _same_pool and not should_force('zone_enriched'):
            print(f'[SKIP] {out.name} — already selected from this pool (same sha256, rule {SELECTION_RULE})')
            continue
        if out.exists():
            print(f'[REFAIT] {out.name} — existing selection comes from another pool or another rule')
            for _stale in (out, _journal, TEMP_SCHEDULED / out.name):
                if _stale.exists():
                    _stale.unlink()
        # 3ter-a — PRE-IMPUTATION OF THE POOL (ticket 029). The v3 selection also balances
        # housing, driving licence and PT pass: these traits must exist on the pool
        # BEFORE the selection, otherwise the margin is empty and the descent ignores it. Same scripts as
        # step 8 (which will replay them identically on the retained ones: deterministic by hashing),
        # in place on the checkpoint. Codes 3 and 4 = not failures (cf. step 8).
        print(f'[PRÉ-IMPUTATION] {pool.name}: equipment, housing and bike traits on the pool…')
        for _label, _module in [('align_minor_traits', 'scripts.data.population.align_minor_traits'),
                                ('enrich_housing_type', 'scripts.data.population.enrich_housing_type'),
                                ('enrich_personal_bike', 'scripts.data.population.enrich_personal_bike'),
                                ('enrich_equipment', 'scripts.data.population.enrich_equipment'),
                                ('align_minor_traits (2e passe)', 'scripts.data.population.align_minor_traits')]:
            _t = time.monotonic()
            _proc = subprocess.run([_select_python, '-m', _module, str(pool)], cwd=str(REPO_ROOT), env=_env,
                                   capture_output=True, text=True)
            _rc = _proc.returncode
            print(f'         {_label:28s} code {_rc} ({time.monotonic() - _t:.0f}s)'
                  + ('' if _rc in (0, 3, 4) else ' ← FAILED: ' + (_proc.stderr or '').strip().split('\n')[-1][:160]))
            if _rc not in (0, 3, 4):
                raise RuntimeError(f'Pre-imputation failed: {_label} (code {_rc})')
        print(f'[SELECT] {SELECT_N} personas from {pool.name} ({pop_size} requested from eqasim)…')
        t0 = time.monotonic()
        proc = subprocess.run(
            [_select_python, '-m', 'scripts.panel.seal_population', 'select',
             '--pool', str(pool), '--n', str(SELECT_N), '--out', str(out)],
            cwd=str(REPO_ROOT), env=_env, capture_output=True, text=True,
        )
        print(proc.stdout.rstrip() or '(pas de sortie)')
        if proc.returncode == 1:
            print('[DÉFICIT] pool too small for at least one cell — carry-overs logged '
                  f'in {out.stem}_selection.json; the chain continues, the final check will see it.')
        elif proc.returncode != 0:
            tail = (proc.stderr or '').strip().split('\n')[-8:]
            raise RuntimeError('Selection failed (code %d):\n%s' % (proc.returncode, '\n'.join(tail)))
        print(f'         {time.monotonic() - t0:.1f}s → Temp/4_zone_enriched/{out.name}')

    # The rest of the chain runs on the selected file, and on it alone.
    POPULATION_SIZES = [SELECT_N]
    POPULATION_TAG = SELECT_TAG
    print()
    print(f'Steps 4 to 9: {pop_filename(SELECT_N)}')

print()
print('Step 3ter finished.')


---
## Step 4 — Travel time computation (scheduling) + Schedule adjustment → `Temp/5_scheduled/`

For each person, we compute **a single travel time** per pair of activities:
- `car` for car owners
- `bicycle` for the others

This time is passed directly to `ajuster_planning` (parameter `travel_times`) — it is **not stored in the JSON**.
Routing is done on the **graphs of the 453-municipality polygon** (`make osmnx-perimeter-graph`,
key `PERIMETER_CACHE_KEY`), not on the 30 km production disc: a 3rd-ring home
has graph nodes of its own there (ticket 031 § 1.4).
The OSMnx SQLite cache (foot / bicycle / car × 24h) is filled separately during the server warm-up.

- **Global deduplication**: pairs shared by several sizes are computed only once.
- **Result**: consistent schedules written directly into `Temp/5_scheduled/`.


In [ ]:
# ── Step 4+5 — Scheduling travel times + Schedule adjustment ─────────────────
import multiprocessing

from population_utils import (
    collect_scheduling_pairs, build_travel_times, ajuster_planning,
)
from route_worker import init_worker, compute_route_worker

print('=' * 60)
print('STEPS 4+5 — Scheduling routes + Schedule adjustment')
print('=' * 60)

to_schedule = []
for pop_size in POPULATION_SIZES:
    fname          = pop_filename(pop_size)
    scheduled_path = TEMP_SCHEDULED / fname

    if not should_force('scheduled') and scheduled_path.exists():
        print(f'[SKIP] {fname} — already in Temp/5_scheduled/')
        continue

    zone_path = TEMP_ZONE / fname
    if not zone_path.exists():
        raise FileNotFoundError(f'zone_enriched file missing — rerun step 3bis: {zone_path}')

    to_schedule.append(pop_size)

if not to_schedule:
    print()
    print('All files are already scheduled — step skipped.')
else:
    all_data: dict[int, list] = {}
    for pop_size in to_schedule:
        all_data[pop_size] = load_json(TEMP_ZONE / pop_filename(pop_size))

    # ── Collecting the scheduling pairs (1 mode per person) ──────────────────
    print()
    print('Collecting the scheduling pairs…')
    global_pairs: set = set()
    per_size_pairs: dict[int, set] = {}
    for pop_size, data in all_data.items():
        pairs = collect_scheduling_pairs(data)
        per_size_pairs[pop_size] = pairs
        global_pairs |= pairs
        print(f'  {pop_filename(pop_size)}: {len(pairs)} pairs (scheduling mode)')

    print(f'Global unique total: {len(global_pairs)} pairs to compute')
    print()

    route_cache: dict[tuple, dict | None] = {}

    if global_pairs:
        # bicycle/foot: hour=None → pass 8h (same result whatever the hour)
        tasks = [
            (lat1, lon1, lat2, lon2, mode, hour if hour is not None else 8)
            for lat1, lon1, lat2, lon2, mode, hour in global_pairs
        ]

        print(f'Computing {len(tasks)} routes with {MAX_WORKERS} workers…')
        t0 = time.monotonic()

        ctx = multiprocessing.get_context('spawn')
        with ProcessPoolExecutor(
            max_workers=MAX_WORKERS,
            mp_context=ctx,
            initializer=init_worker,
            initargs=(LLMAGENTS_PATH, str(OSMNX_CACHE), CACHE_KEY),
        ) as pool:
            results = list(tqdm(
                pool.map(compute_route_worker, tasks, chunksize=20),
                total=len(tasks),
                desc='scheduling routes',
            ))

        elapsed = time.monotonic() - t0
        print(f'Done in {elapsed:.1f}s  ({elapsed / len(tasks) * 1000:.1f} ms/route)')

        ok_count = null_count = 0
        for args, result in results:
            lat1, lon1, lat2, lon2, mode, actual_hour = args
            route_cache[(lat1, lon1, lat2, lon2, mode, actual_hour if mode == 'car' else None)] = result
            if result is None:
                null_count += 1
            else:
                ok_count += 1
        print(f'Routes: {ok_count} OK, {null_count} not routable (None)')
        print()

    # ── Build travel_times + ajuster_planning + save ────────────────────────
    for pop_size in to_schedule:
        fname = pop_filename(pop_size)
        data  = all_data[pop_size]

        travel_times_list = build_travel_times(data, route_cache)

        sched_errors = 0
        for entry, tt in zip(data, travel_times_list):
            acts = entry.get('identity', {}).get('activities', [])
            try:
                entry['identity']['activities'] = ajuster_planning(
                    fname, entry.get('person_id', '?'), acts,
                    travel_times=tt, raise_error=True,
                )
            except ValueError as exc:
                sched_errors += 1

        save_json(data, TEMP_SCHEDULED / fname)
        n_p, n_e = check_temporal_order(data)
        status = '[OK]  ' if n_e == 0 else '[WARN]'
        print(f'[OK]   {fname} — {len(data)} persons → Temp/5_scheduled/')
        print(f'       {status} Temporal order: {len(data) - n_p}/{len(data)} valid')
        if sched_errors:
            print(f'       [WARN] {sched_errors} scheduling conflicts left unsettled')

print()
print('Steps 4+5 finished.')

---
## Step 5 — *(merged with step 4)*

Schedule adjustment is now done directly in step 4.
This cell is kept for pipeline compatibility but runs nothing.


In [ ]:
# Step 5 merged with step 4 — nothing to do here.
print("Step 5 — already done in step 4.")


---
## Step 6 — OSMnx SQLite warm-up → `data/cache/osmnx/toulouse_population_N/`

For each unique O-D pair of the population, we compute **all the routes** needed by the server:
- `foot` × 1 entry (time-independent)
- `bicycle` × 1 entry (time-independent)
- `car` × 24 hourly entries (TomTom congestion per time slot)

The results are written into the persistent SQLite cache (`osmnx_cache.db`).
At server startup, `get_direct_plan()` will find **100 % hits** in this cache — provided that the mount `./data/cache/osmnx:/app/data/cache/osmnx` of the `controller` service is in place (checked by the safeguard of the paths cell) and that `routing_version` (`config/terminal_time.yaml`) has not moved since the warm-up: a row from an outdated version is never served again.

The cell is **idempotent**: routes already in the database are skipped.


In [ ]:
# ── Step 6 — OSMnx SQLite warm-up ───────────────────────────────────────────
import multiprocessing
from datetime import datetime as _dt, date as _date

from population_utils import collect_warmup_pairs
from route_worker import init_worker, compute_route_worker
from trip_helper.osmnx_persistent_cache import OsmnxPersistentCache

_SIM_DATE = _date(2024, 1, 8)  # Monday — must match the simulation day

print('=' * 60)
print('STEP 6 — OSMnx SQLite warm-up (foot + bicycle + car × 24h)')
print('=' * 60)

if SKIP_WARMUP:
    print('[SKIP] SKIP_WARMUP = True — OSMnx warm-up skipped; the runtime will compute the missing')
    print('       itineraries on demand. Rerun with SKIP_WARMUP = False (and MAX_WORKERS suited to the RAM).')
    POPULATION_SIZES_WARMUP = []
else:
    POPULATION_SIZES_WARMUP = list(POPULATION_SIZES)

for pop_size in POPULATION_SIZES_WARMUP:
    fname          = pop_filename(pop_size)
    scheduled_path = TEMP_SCHEDULED / fname

    if not scheduled_path.exists():
        print(f'[SKIP] {fname} — scheduled file missing, rerun step 4+5')
        continue

    data      = load_json(scheduled_path)
    all_pairs = collect_warmup_pairs(data)  # foot/bicycle × 1 + car × 24h

    _cache_dir = OSMNX_ROUTE_CACHE / Path(fname).stem
    _sqlite    = OsmnxPersistentCache(str(_cache_dir))

    # ── Coverage check (idempotence) ─────────────────────────────────────────
    pair_list = list(all_pairs)
    missing   = []
    for lat1, lon1, lat2, lon2, mode, hour in pair_list:
        actual_h  = hour if hour is not None else 8
        cdt       = _dt(_SIM_DATE.year, _SIM_DATE.month, _SIM_DATE.day, actual_h, 0)
        key, *_   = OsmnxPersistentCache.make_key(cdt, mode, lat1, lon1, lat2, lon2)
        if not _sqlite.lookup(key).found:
            missing.append((lat1, lon1, lat2, lon2, mode, hour))

    print(f'\n{fname}: {len(all_pairs)} pairs in total — {len(missing)} to compute')

    if not missing:
        print(f'  [OK] SQLite cache already complete.')
        continue

    # ── Computing the missing routes ──────────────────────────────────────────
    tasks = [
        (lat1, lon1, lat2, lon2, mode, hour if hour is not None else 8)
        for lat1, lon1, lat2, lon2, mode, hour in missing
    ]

    print(f'  Computing {len(tasks)} routes with {MAX_WORKERS} workers…')
    t0 = time.monotonic()

    ctx = multiprocessing.get_context('spawn')
    with ProcessPoolExecutor(
        max_workers=MAX_WORKERS,
        mp_context=ctx,
        initializer=init_worker,
        initargs=(LLMAGENTS_PATH, str(OSMNX_CACHE), CACHE_KEY),
    ) as pool:
        results = list(tqdm(
            pool.map(compute_route_worker, tasks, chunksize=20),
            total=len(tasks),
            desc=f'warmup {pop_size}',
        ))

    elapsed = time.monotonic() - t0
    print(f'  Done in {elapsed:.1f}s  ({elapsed / len(tasks) * 1000:.1f} ms/route)')

    # ── Writing to SQLite ──────────────────────────────────────────────────────
    n_ok = n_null = 0
    for (lat1, lon1, lat2, lon2, mode, hour), (_, result) in zip(missing, results):
        actual_h              = hour if hour is not None else 8
        cdt                   = _dt(_SIM_DATE.year, _SIM_DATE.month, _SIM_DATE.day, actual_h, 0)
        key, date_s, dow, bkt = OsmnxPersistentCache.make_key(cdt, mode, lat1, lon1, lat2, lon2)
        _sqlite.store(key, date_s, dow, bkt, mode, lat1, lon1, lat2, lon2, result)
        if result is None:
            n_null += 1
        else:
            n_ok += 1

    print(f'  SQLite: {n_ok} OK, {n_null} not routable → {_cache_dir.relative_to(REPO_ROOT)}/osmnx_cache.db')

print()
print('Step 6 finished.')


---
## Final export → `data/eqasim_output/`

Copy of the scheduled files from `Temp/5_scheduled/` to the final output folder `data/eqasim_output/`. A quality summary is shown for each size before the copy:

| Indicator | Meaning |
|---|---|
| `missing_routes` | Agents with at least one trip that could not be computed (inaccessible zones) |
| `missing_any_mode` | Agents whose transport mode remains undetermined |
| `missing_pt` | Agents attached to PT but without a Tisséo stop within the 1,500 m radius |

The exported files can be consumed directly by the GAMA simulation and the LLM agent server.

In [ ]:
# check_enrichment removed — routes are no longer in the JSON
for pop_size in POPULATION_SIZES:
    fname = pop_filename(pop_size)
    path = TEMP_SCHEDULED / fname
    if not path.exists():
        print(f'[MISSING] {fname}')
        continue
    data = load_json(path)
    n_p, n_e = check_temporal_order(data)
    print(f'{fname}: {len(data)} persons, {len(data)-n_p}/{len(data)} valid schedules')


---
## Step 7 — Final export `Temp/5_scheduled/` → `data/population/`

**This is the step the notebook was missing**, and its absence was silent: the five
previous steps worked in `Temp/`, but nothing copied the result to
`data/population/`, the only folder read by GAMA and by the agent server (mounted in the
containers under `/eqasim-output`). The file lying there was therefore the **raw output
of eqasim** dropped by step 1 — without activity correction, without PT flag, and above all
**without readjusted schedules**: `toulouse_population_1000.json` had 0 scheduled activities
where `Temp/5_scheduled/` counts 3,944.

The export checks before copying, and **refuses** rather than overwriting a valid
population with a degraded one: fewer agents than the raw source, no scheduled
activity, or an invalid temporal order make the step fail.

A `.bak` backup of the previous version is kept next to the file.


In [ ]:
# ── Step 7 — Final export → data/population/ ─────────────────────────────────
import shutil

print('=' * 60)
print('STEP 7 — Final export → data/population/')
print('=' * 60)

EXPORTED = []
for pop_size in POPULATION_SIZES:
    fname = pop_filename(pop_size)
    src   = TEMP_SCHEDULED / fname
    dst   = POP_DIR / fname

    if not src.exists():
        print(f'[MANQUE] {fname} missing from Temp/5_scheduled/ — step 4+5 not run?')
        continue

    data = load_json(src)
    n_agents     = len(data)
    n_activities = sum(len(p['identity']['activities']) for p in data)
    n_scheduled  = sum(1 for p in data for a in p['identity']['activities']
                       if a.get('scheduled_start_time') is not None)
    n_pt         = sum(1 for p in data for a in p['identity']['activities']
                       if (a.get('location') or {}).get('public_transport'))
    n_bad, _     = check_temporal_order(data)

    # Safeguards: a population is only overwritten by a better one.
    problems = []
    if n_scheduled == 0:
        problems.append('aucune activité planifiée (étape 5 non jouée)')
    if n_bad:
        problems.append(f'{n_bad} planning(s) temporellement invalides')
    raw = TEMP_RAW / fname
    if raw.exists():
        n_raw = len(load_json(raw))
        if n_agents < n_raw * 0.5:
            problems.append(f'{n_agents} agents contre {n_raw} au brut — perte massive')
    if problems:
        print(f'[REFUS]  {fname} : ' + ' ; '.join(problems))
        print(f'         {dst.name} left in place, nothing was overwritten.')
        continue

    if dst.exists():
        backup = dst.with_suffix('.json.bak')
        shutil.copy2(dst, backup)

    # Byte copy: we preserve exactly what step 5 wrote.
    shutil.copy2(src, dst)
    EXPORTED.append(fname)
    print(f'[OK]     {fname}  {n_agents} agents, {n_activities} activities, '
          f'{n_scheduled} scheduled, {n_pt} served by PT')
    print(f'         → {dst.relative_to(REPO_ROOT)}')

print()
print(f'Step 7 finished — {len(EXPORTED)}/{len(POPULATION_SIZES)} file(s) exported.')


---
## Step 8 — Traits imputed from the EMC² microdata

Four persona traits come **neither** from eqasim **nor** from the INSEE tables: they are
read or imputed from the EMC² Toulouse 2023 survey (ProGEDO `lil-1750`), or corrected after
the fact.
Without this step, the population is syntactically valid and **statistically wrong** —
precisely the case that produced an inverted bike-ownership gradient for
months.

| Script | Trait | What it corrects |
|---|---|---|
| `align_minor_traits` | `has_driving_license`, purposes, `car_availability` | children's licences, inconsistent purposes, cars "to share" because of a minor's licence |
| `enrich_housing_type` | `housing_type` | trait **absent** from eqasim; imputed by (fine zone, household size) |
| `enrich_personal_bike` | `personal_bike` | number of bikes **copied** from an ENTD 2008 household matched without the household size → inverted gradient |
| `enrich_residence_zone` | `residence_zone`, `residence_commune` | ring **guessed from the distance** to the Capitole instead of being read from the survey's municipal breakdown → 24.4 % of personas compared with the target of another zone (ticket 021) |

**Order matters**: `enrich_personal_bike` reads `housing_type` for its
validation report (equipment × housing type cross), so it runs after
`enrich_housing_type`. `enrich_residence_zone` depends on nothing and runs first. All
four are **idempotent** — replaying them costs nothing and changes nothing.

**A trait apart**: `residence_zone` is **observed**, not imputed — a home is in
a municipality or it is not. No draw, no distribution, no salt: its validation therefore
concerns not a distribution but an **agreement** between the classification by fine zone code and
the classification by geometric membership. Its exit code `4` is not a failure: it says
that the gates pass but that the population is spatially more concentrated than the
framing — axis A9 of ticket 020, which is corrected in the **draw**, not here.

Two principles, inherited from tickets 015 and 019:

- **no silent fallback**: a script whose restricted-access resource is missing
  or stale **refuses to run** instead of imputing blindly. The resources are
  (re)produced with `make zones`, `make housing-type`, `make bike-ownership`;
- **a failure here is not a warning**: the population is then **incomplete**, and
  the step 9 audit says so without hedging.


In [ ]:
# ── Step 8 — Traits imputed from EMC² ────────────────────────────────────────
import subprocess

print('=' * 60)
print('STEP 8 — Traits imputed from the EMC² microdata')
print('=' * 60)

# Post-processing interpreter. They import `mobility_core` (shapely, pyproj,
# geopandas) and `scripts.*`: the notebook kernel does not necessarily have these dependencies,
# so we prefer the project venv, as the Makefile does (SYNTHESIS_PYTHON).
_venv = REPO_ROOT / 'services' / 'llm-agents' / '.venv' / 'bin' / 'python'
POST_PROCESS_PYTHON = str(_venv) if _venv.exists() else sys.executable
print(f'Interpreter: {POST_PROCESS_PYTHON}')

# Imposed order, and two constraints rather than one:
#
# 1. `enrich_personal_bike` reads `housing_type`. `enrich_residence_zone` depends on
#    nothing and runs first among the enrichments.
# 2. `car_availability` DERIVES from the number of licences in the household (rule 4 of
#    `align_minor_traits`). `enrich_equipment` rewrites the licences: any
#    `car_availability` computed BEFORE it is stale, and nothing would flag it.
#    Hence the second pass of `align_minor_traits` — it is idempotent, replaying it only
#    costs the recomputation — then a read-only check. This is
#    the explicit warning of ticket 017; reversing it is the trap it describes.
POST_STEPS = [
    ('align_minor_traits',   'scripts.data.population.align_minor_traits',   []),
    ('enrich_residence_zone', 'scripts.data.population.enrich_residence_zone', ['--check']),
    ('enrich_housing_type',  'scripts.data.population.enrich_housing_type',  ['--check']),
    ('enrich_personal_bike', 'scripts.data.population.enrich_personal_bike', ['--check']),
    # Tickets 016 and 017: PT pass and driving licence, learnt on EMC² (`make
    # equipment-propensity`). Only sets them here — the acceptance check comes after the 2nd pass.
    ('enrich_equipment',     'scripts.data.population.enrich_equipment',     []),
    # 2nd pass: recomputes `car_availability` on the licences that have just been set.
    ('align_minor_traits (2e passe)', 'scripts.data.population.align_minor_traits', []),
    # READ-ONLY acceptance check, once `car_availability` is back in agreement. The code 4
    # it may return is not a failure of the trait: cf. the note below.
    ('enrich_equipment (recette)', 'scripts.data.population.enrich_equipment',
     ['--dry-run', '--check']),
]

targets = [POP_DIR / pop_filename(n) for n in POPULATION_SIZES]
targets = [p for p in targets if p.exists()]
if not targets:
    raise FileNotFoundError(
        f'No population in {POP_DIR} — run step 7 (final export) first.')

_env = {**os.environ, 'PYTHONPATH': str(REPO_ROOT)}
POST_RESULTS = {}
for label, module, extra in POST_STEPS:
    print(f'\n── {label} ' + '─' * (56 - len(label)))
    proc = subprocess.run(
        [POST_PROCESS_PYTHON, '-m', module, *[str(p) for p in targets], *extra],
        cwd=str(REPO_ROOT), env=_env, capture_output=True, text=True,
    )
    print(proc.stdout.rstrip() or '(pas de sortie)')
    if proc.returncode != 0:
        # We do NOT interrupt the loop: each trait is independent, and knowing
        # which ones are missing is better than stopping at the first one.
        #
        # Code 3 is NOT a failure: it says "enriched, but not enough households
        # to decide" — the normal case of populations of 10 or 100 agents, where a
        # cross-tabulation cannot arbitrate anything. Confusing it with a real failure teaches
        # one to ignore failures.
        if proc.returncode == 3:
            print('[NON VALIDÉ] population too small to decide — not a failure')
        elif proc.returncode == 4:
            # Code 4 is not a failure either: the trait's gates pass, but
            # the COMPOSITION of the population departs from the survey. Two cases
            # return it: the spatial distribution (axis A9 of ticket 020) and, since
            # `enrich_equipment`, an occupation stratum whose covariates are not
            # those of the survey. Both are corrected in the DRAW, not in
            # the enrichment.
            print('[ÉCART CADRAGE] gates passed, composition of the population '
                  'departs from the framing — corrected at the draw, not a defect of the trait')
        else:
            print(f'[ÉCHEC] code {proc.returncode}')
            tail = (proc.stderr or '').strip().split('\n')[-6:]
            for line in tail:
                print(f'        {line}')
    POST_RESULTS[label] = proc.returncode

print()
_verdict = {0: 'ok', 3: 'non validé (trop petit)', 4: 'ok (écart de cadrage A9)'}
print('Step 8 finished — ' + ' | '.join(
    f'{k}: {_verdict.get(v, f"ÉCHEC({v})")}' for k, v in POST_RESULTS.items()))

# What must interrupt the chain: a trait that could not be set at all.
_blocking = {k: v for k, v in POST_RESULTS.items() if v not in (0, 3, 4)}
if _blocking:
    print(f'\n[BLOQUANT] {", ".join(_blocking)} — the population is INCOMPLETE.')
    print('           Read the cause above: it says which resource to produce')
    print('           (make zones / make housing-type / make bike-ownership /')
    print('            make communes-couronnes).')


---
## Step 9 — Audit: is the population complete?

Last cell, and the only one that pronounces a verdict. It rereads the files
actually dropped into `data/population/` and checks the presence of the traits that the
simulation consumes — not their statistical correctness (that is the job of the `--check` of
step 8), but their **presence**.

The verdict is binary and deliberately severe: a trait missing on a significant
share of the agents makes the population **unusable**, because downstream consumers
do not all complain about it. A missing `personal_bike` was treated as a "normal
bike" until ticket 015 — 100 % of agents on bikes, without a single log line.


In [ ]:
# ── Step 9 — Completeness audit ──────────────────────────────────────────────
print('=' * 60)
print('STEP 9 — Completeness audit')
print('=' * 60)

# Traits expected in traits_json, and the minimum acceptable coverage.
# `housing_type` and `personal_bike` are legitimately missing outside the EMC² fine-zone
# layer (~5 % of homes, clearly outer municipalities): the threshold accounts for it.
EXPECTED_TRAITS = {
    'age':                 1.00,
    'gender':              1.00,
    'household_size':      1.00,
    'main_occupation':     1.00,
    'number_of_cars':      1.00,
    'has_driving_license': 1.00,
    'has_pt_subscription': 1.00,
    'housing_type':        0.90,
    'personal_bike':       0.90,
}

verdicts = {}
for pop_size in POPULATION_SIZES:
    fname = pop_filename(pop_size)
    path  = POP_DIR / fname
    if not path.exists():
        print(f'\n[ABSENT] {fname} — not exported')
        verdicts[fname] = False
        continue

    data = load_json(path)
    n = len(data)
    n_scheduled = sum(1 for p in data for a in p['identity']['activities']
                      if a.get('scheduled_start_time') is not None)
    print(f'\n{fname} — {n} agents, {n_scheduled} scheduled activities')

    ok = n_scheduled > 0
    if not ok:
        print('  [ÉCHEC] no scheduled activity: step 5 or 7 not run')
    for trait, threshold in EXPECTED_TRAITS.items():
        present = sum(1 for p in data
                      if (p['identity'].get('traits_json') or {}).get(trait) is not None)
        share = present / n if n else 0.0
        good = share >= threshold
        ok &= good
        flag = 'ok   ' if good else 'ÉCHEC'
        print(f'  {flag} {trait:22s} {present:6d}/{n} ({100 * share:5.1f} % '
              f'— threshold {100 * threshold:.0f} %)')
    verdicts[fname] = bool(ok)

print()
print('=' * 60)
for fname, ok in verdicts.items():
    print(f'{"POPULATION COMPLÈTE  " if ok else "POPULATION INCOMPLÈTE"}  {fname}')
print('=' * 60)
if not all(verdicts.values()):
    print('\nA missing trait is not a detail: downstream consumers do not all')
    print('complain about it. Go back to step 8 and read the exact cause — it says')
    print('which restricted-access resource to produce (make zones / housing-type /')
    print('bike-ownership).')
